In [228]:
import pandas as pd
import os
import math

In [229]:
if os.path.exists(r'temp\temp_data.csv'):
    data = pd.read_csv(r'temp\temp_data.csv')
    print('Utilizando datos temp')

else:
    data = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSL8e5uoUExt5a-LDPCw0rEcFTm0SqAhLz8sYT8sbkYtse1pvMHY9Qij547diNhlP__DYxtuT8XojRO/pub?gid=1596580014&single=true&output=csv')
    os.mkdir('temp')
    data.to_csv(r'temp\temp_data.csv')
    print('Carpeta temp con datos temp creada')
    

Utilizando datos temp


C:\Users\Admin\AppData\Local\Temp\ipykernel_15724\68320909.py:2: DtypeWarning: Columns (0: Comments, 1: Month, 2: Year, 3: SOM, 4: Unnamed: 21) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(r'temp\temp_data.csv')


In [230]:
data['datestamp'] = pd.to_datetime(data['datestamp'])

In [231]:
start_date = '20260601'
end_date = '20260630'

In [232]:
# Iniciamos con los datos que conforman 1ro de junio 2026 hasta el 30 de junio 2026

Base_data = data.loc[
    
    (data['datestamp'] >= start_date)
    &
    (data['datestamp'] <= end_date)
          
         
         ]

In [233]:
Base_data['LOB'].unique()

<ArrowStringArray>
[              'ADMINISTRATION',                    'ANALYTICS',
           'COMPLIANCE CONTROL',        'CUSTOMER SATISFACTION',
                 'DATA SCIENCE',                      'FINANCE',
                    'FRAUD/AML',                 'GO TO MARKET',
         'HUMAN RESOURCES (HR)',               'INBOUND & CHAT',
  'INFORMATION TECHNOLOGY (IT)',       'LEADERSHIP DEVELOPMENT',
               'MULTIFUNCTIONS',                         'OFAC',
                   'ONBOARDING',                   'OPERATIONS',
 'PORTFOLIO PERFORMANCE & RISK',                      'PROCESS',
                      'PRODUCT',                           'QA',
      'RELATIONSHIP MANAGEMENT',           'REVENUE OPERATIONS',
            'RISK & COMPLIANCE',                        'SALES',
                   'SERVICINGS',                 'UNDERWRITING']
Length: 26, dtype: str

In [234]:
lob = 'DATA SCIENCE'

In [235]:
# Aqui filtro los datos para solo quedarme con los datos necesarios- Fecha cop

working_data = Base_data.loc[
    
    (Base_data['datestamp'] >= start_date) # La fecha de inicio
    &
    (Base_data['datestamp'] <= end_date) # Fecha final
    &
    (Base_data['LOB'] == lob)     # Aqui está el Departamento          
         
         ]

In [236]:
fechas_excluir = pd.to_datetime([
    '2026-06-06', 
    '2026-06-13', 
    '2026-06-20', 
    '2026-06-27'
]).date


Lunch_Data = working_data.loc[
    
    (~working_data['Status'].isin(['Vacation','Maternity/Paternity Leave', 'No Show', 'Medical License', 'Sick Leave']))
    &
    (~working_data['datestamp'].dt.date.isin(fechas_excluir))
                 
                 ]

Lunch_Data['Lunch'] = pd.to_timedelta(Lunch_Data['Lunch'])

In [237]:
away_base_data = working_data.loc[
    
    (~working_data['Status'].isin(['Vacation','Maternity/Paternity Leave', 'Medical License', 'No Show', 'Sick Leave']))
   
   ]

away_base_data['away'] = pd.to_timedelta(away_base_data['away'])

In [238]:
conformance_data = working_data.loc[
    
    (~working_data['Status'].isin(['Maternity/Paternity Leave','Maternity/Paternity Leave', 'Medical License', 'Sick Leave']))
                 ]

conformance_data['Total work time'] = pd.to_timedelta(conformance_data['Total work time'])


In [239]:
def punctuality(name):

    ##### Lateness Score #####
    punct_data = working_data.loc[
        (working_data['Full Name'] == name)
        &
        (~working_data['Status'].isin(['Vacation', 'Maternity/Paternity Leave', 'Medical License', 'Sick Leave']))
    
    ]
    
    total_records = punct_data['Clock in time'].size

    if total_records == 0:
        return 0

    not_allowed = punct_data.loc[punct_data['Status'].isin(['Late', 'Called Out', 'No Show']), 'Status'].size
    results = round((not_allowed / total_records) * 100)
    new_result = 100 - results
    return (new_result / 100) * 5

    

In [240]:
def conformance(name):
    conf_data = conformance_data.loc[conformance_data['Full Name'] == name                         
                            
                            ]
    conformance_dias_laborados = conf_data['Full Name'].size
    if conformance_dias_laborados ==0:
        return 0
    horas_esperadas = conformance_dias_laborados * pd.Timedelta(hours=8)
    actual_horas_trabajadas = conf_data['Total work time'].sum()
    score_horas_trabajadas = round(((actual_horas_trabajadas/horas_esperadas)/1)*5, 2)


##### Lunch Score #####
    Lunch_Data_ = Lunch_Data.loc[Lunch_Data['Full Name'] == name]
   
    dias_laborados = Lunch_Data_['Lunch'].size
    if dias_laborados == 0:
        return 0
    lunch_esperado = dias_laborados * pd.Timedelta(hours=1)
    lunch_tomado = Lunch_Data_['Lunch'].sum()
    Resultado = lunch_esperado / lunch_tomado
    resultado_lunch = (Resultado / 1) * 5

    ##### Away Score #####
    away_data = away_base_data.loc[away_base_data['Full Name'] == name]
    dias_ = away_data['Full Name'].size
    away_esperado = dias_ * pd.Timedelta(minutes=15)
    actual_away = away_data['away'].sum()
    resultado_away = ((actual_away / away_esperado) / 1) * 5

    if resultado_away <= 5:
        score_away = 5
    else:
        resultado_away_final = resultado_away - 5
        score_away = 5 - resultado_away_final

    grouped_score = score_horas_trabajadas + resultado_lunch + score_away

    return round(grouped_score / 3,2)


In [241]:
# Puntuality Column

#Grouping the data for calculation
grouped_data = working_data.groupby(['Full Name']).agg(grouped = ('Full Name', 'first'))

#Adding columns using functions
grouped_data['Punctuality_score'] = grouped_data['grouped'].apply(punctuality)
grouped_data['Conformance'] = grouped_data['grouped'].apply(conformance)



grouped_data = grouped_data.reset_index()
grouped_data = grouped_data[['Full Name', 'Punctuality_score', 'Conformance']]

grouped_data = grouped_data.sort_values(by='Punctuality_score', ascending=False)

#------------------------------------------------------------------------------------



In [242]:
grouped_data = grouped_data[['Full Name', 'Punctuality_score', 'Conformance']]


In [243]:
grouped_data

,Full Name,Punctuality_score,Conformance
0,JUAN MIGUEL GARCIA VASQUEZ,5.0,5.01


In [244]:
grouped_data.dtypes

Full Name                str
Punctuality_score    float64
Conformance          float64
dtype: object

## Debugging

In [245]:
# Debugging Adrian Anibal solando

In [246]:
# # Punctuality Score

punct_data = working_data.loc[
    (working_data['Full Name'] == 'MARCOS JOSE JIMENEZ NUÑEZ')
    &
    (~working_data['Status'].isin(['Vacation', 'Maternity/Paternity Leave', 'Medical License', 'Sick Leave']))
]



In [247]:
total_records = punct_data['Clock in time'].size





In [248]:
total_records

0

In [249]:
not_allowed = punct_data.loc[punct_data['Status'].isin(['Late', 'Called Out', 'No Show']), 'Status'].size


In [250]:
not_allowed

0

In [251]:
results = round((not_allowed / total_records) * 100)


ZeroDivisionError: division by zero

In [ ]:
results

10

In [ ]:
new_result = 100 - results


In [ ]:
new_result

90

In [ ]:
resull=  (new_result / 100) * 5

In [ ]:
resull

4.5

In [ ]:
# Lunch Score

Lunch_Data_ = Lunch_Data.loc[Lunch_Data['Full Name'] == 'MARCOS JOSE JIMENEZ NUÑEZ']

dias_laborados = Lunch_Data_['Lunch'].size




In [ ]:
dias_laborados

19

In [ ]:
lunch_esperado = dias_laborados * pd.Timedelta(hours=1)


In [ ]:
lunch_esperado

Timedelta('0 days 19:00:00')

In [ ]:
lunch_tomado = Lunch_Data_['Lunch'].sum()


In [ ]:
lunch_tomado

Timedelta('0 days 19:26:27')

In [ ]:
Resultado = lunch_esperado / lunch_tomado


In [ ]:
Resultado

0.97732436023833

In [ ]:
resultado_lunch = (Resultado / 1) * 5

In [ ]:
resultado_lunch

4.88662180119165

In [ ]:
conf_data = conformance_data.loc[conformance_data['Full Name'] == 'MARCOS JOSE JIMENEZ NUÑEZ'                         
                        
                        ]



In [ ]:
conformance_dias_laborados = conf_data['Full Name'].size


In [ ]:
conformance_dias_laborados

23

In [ ]:

horas_esperadas = conformance_dias_laborados * pd.Timedelta(hours=8)


In [ ]:
horas_esperadas

Timedelta('7 days 16:00:00')

In [ ]:
actual_horas_trabajadas = conf_data['Total work time'].sum()


In [ ]:
actual_horas_trabajadas

Timedelta('6 days 16:37:42')

In [ ]:
score_horas_trabajadas = round(((actual_horas_trabajadas/horas_esperadas)/1)*5, 2)

In [ ]:
score_horas_trabajadas

4.36

In [ ]:
# #Away Score

away_data = away_base_data.loc[away_base_data['Full Name'] == 'MARCOS JOSE JIMENEZ NUÑEZ']


In [ ]:
dias_ = away_data['Full Name'].size


In [ ]:
dias_

21

In [ ]:
away_esperado = dias_ * pd.Timedelta(minutes=15)


In [ ]:
away_esperado

Timedelta('0 days 05:15:00')

In [ ]:
actual_away = away_data['away'].sum()


In [ ]:
actual_away

Timedelta('0 days 02:08:52')

In [ ]:
resultado_away = ((actual_away / away_esperado) / 1) * 5



In [ ]:
resultado_away

2.0455026455026455

In [ ]:
if resultado_away <= 5:
    score_away = 5
else:
    resultado_away_final = resultado_away - 5
    score_away = 5 - resultado_away_final

In [ ]:
score_away

5